In [7]:
import xarray as xr
import pandas as pd
import numpy as np
import geopandas as gpd
import json
import dask
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

from xcube.core.store import new_data_store
from xcube.core.chunk import chunk_dataset
from xcube.core.gridmapping import GridMapping
from xcube.core.geom import mask_dataset_by_geometry
from xcube_resampling.spatial import resample_in_space
from xcube_resampling.gridmapping import GridMapping
from dask.distributed import Client, LocalCluster

In [8]:
INPUT_DIR = "input_irrigation_10years"

In [9]:
irr_store = new_data_store("file", root=INPUT_DIR)

In [10]:
# bbox = [-5, 40, 3, 44] # Ebro Basin
# time_range = ("2020-01-01", "2021-12-31")
# time_range = ("2020-01-01", "2020-01-31")
bbox = [-31, 27, 40, 81] # Europe
time_range = ("2016-01-01", "2025-09-30")

In [11]:
json_file_path = "credentials.json"
with open(json_file_path, "r") as j:
    credentials = json.loads(j.read())

In [12]:
import logging
logging.getLogger().setLevel(logging.WARNING)
LOG = logging.getLogger("xcube.clms")
LOG.handlers.clear() 
LOG.setLevel(logging.DEBUG)

handler = logging.StreamHandler()
handler.setLevel(logging.DEBUG)
handler.setFormatter(logging.Formatter(
    "%(asctime)s [%(levelname)s] %(name)s: %(message)s"
))
LOG.addHandler(handler)

In [13]:
%%time
clms_data_store = new_data_store("clms", credentials=credentials)

2025-10-07 12:45:49,372 [INFO] xcube.clms: Fetching datasets metadata from https://land.copernicus.eu/api
2025-10-07 12:45:49,373 [DEBUG] xcube.clms: Making a request to https://land.copernicus.eu/api/@search/?portal_type=DataSet&fullobjects=1
2025-10-07 12:45:52,427 [DEBUG] xcube.clms: Making a request to https://land.copernicus.eu/api/@search/?b_start=25&portal_type=DataSet&fullobjects=1
2025-10-07 12:45:54,854 [DEBUG] xcube.clms: Making a request to https://land.copernicus.eu/api/@search/?b_start=50&portal_type=DataSet&fullobjects=1
2025-10-07 12:45:56,114 [DEBUG] xcube.clms: Making a request to https://land.copernicus.eu/api/@search/?b_start=75&portal_type=DataSet&fullobjects=1
2025-10-07 12:45:57,426 [DEBUG] xcube.clms: Making a request to https://land.copernicus.eu/api/@search/?b_start=100&portal_type=DataSet&fullobjects=1
2025-10-07 12:45:59,576 [DEBUG] xcube.clms: Making a request to https://land.copernicus.eu/api/@search/?b_start=125&portal_type=DataSet&fullobjects=1
2025-10-0

CPU times: user 265 ms, sys: 83.3 ms, total: 348 ms
Wall time: 27.7 s


In [14]:
%%time
clms_data = clms_data_store.open_data("daily-surface-soil-moisture-v1.0", time_range=time_range)
clms_data

2025-10-07 12:46:17,046 [DEBUG] xcube.clms: Token expired or not present. Refreshing token.
2025-10-07 12:46:17,077 [DEBUG] xcube.clms: Making a request to https://land.copernicus.eu/@@oauth2-token
2025-10-07 12:46:17,373 [DEBUG] xcube.clms: Token refreshed successfully.
2025-10-07 12:46:17,375 [DEBUG] xcube.clms: Token expired or not present. Refreshing token.
2025-10-07 12:46:17,404 [DEBUG] xcube.clms: Making a request to https://land.copernicus.eu/@@oauth2-token
2025-10-07 12:46:17,599 [DEBUG] xcube.clms: Token refreshed successfully.
2025-10-07 12:46:17,599 [DEBUG] xcube.clms: Current token valid. Reusing it.
2025-10-07 12:46:17,600 [DEBUG] xcube.clms: Making a request to https://land.copernicus.eu/api/@get-download-file-urls/?dataset_uid=c073cf8a1d594593bc5d2f9024f0dc60&download_information_id=0524bf44-6624-4e94-b88a-38af27388b31&date_from=2014-01-01&date_to=2025-10-07
2025-10-07 12:46:18,708 [DEBUG] xcube.clms: Processing 3516 files in batches of 20
2025-10-07 12:46:18,709 [DEBUG

urls:: ['https://globalland.vito.be/download/netcdf/surface_soil_moisture/ssm_1km_v1_daily/2016/20160101/c_gls_SSM1km_201601010000_CEURO_S1CSAR_V1.1.1.nc', 'https://globalland.vito.be/download/netcdf/surface_soil_moisture/ssm_1km_v1_daily/2016/20160102/c_gls_SSM1km_201601020000_CEURO_S1CSAR_V1.1.1.nc', 'https://globalland.vito.be/download/netcdf/surface_soil_moisture/ssm_1km_v1_daily/2016/20160103/c_gls_SSM1km_201601030000_CEURO_S1CSAR_V1.1.1.nc', 'https://globalland.vito.be/download/netcdf/surface_soil_moisture/ssm_1km_v1_daily/2016/20160104/c_gls_SSM1km_201601040000_CEURO_S1CSAR_V1.1.1.nc', 'https://globalland.vito.be/download/netcdf/surface_soil_moisture/ssm_1km_v1_daily/2016/20160105/c_gls_SSM1km_201601050000_CEURO_S1CSAR_V1.1.1.nc', 'https://globalland.vito.be/download/netcdf/surface_soil_moisture/ssm_1km_v1_daily/2016/20160106/c_gls_SSM1km_201601060000_CEURO_S1CSAR_V1.1.1.nc', 'https://globalland.vito.be/download/netcdf/surface_soil_moisture/ssm_1km_v1_daily/2016/20160107/c_gls_S

/home/yogesh/Projects/BC/xcube-clms/xcube_clms/utils.py:691: FutureWarning: In a future version of xarray the default value for data_vars will change from data_vars='all' to data_vars=None. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set data_vars explicitly.
  ds = xr.open_mfdataset(batch_paths, engine=engine, **kwargs)
2025-10-07 12:46:31,724 [DEBUG] xcube.clms: Successfully opened batch of 20 files
2025-10-07 12:46:31,725 [DEBUG] xcube.clms: Processing batch 2 (files 21-42) with batch size 22
2025-10-07 12:46:31,725 [DEBUG] xcube.clms: Attempting to open batch of 22 files (attempt 1/10)
/home/yogesh/Projects/BC/xcube-clms/xcube_clms/utils.py:691: FutureWarning: In a future version of xarray the default value for data_vars will change from data_vars='all' to data_vars=None. This is likely to lead t

CPU times: user 6min 53s, sys: 2min 11s, total: 9min 5s
Wall time: 1h 31min 37s


<xarray.Dataset> Size: 796GB
Dimensions:    (time: 3516, lat: 4144, lon: 6832)
Coordinates:
  * lat        (lat) float64 33kB 72.0 71.99 71.98 71.97 ... 35.02 35.01 35.0
  * lon        (lon) float64 55kB -11.0 -10.99 -10.98 ... 49.98 49.99 50.0
  * time       (time) datetime64[ns] 28kB 2016-01-01 2016-01-02 ... 2025-09-30
Data variables:
    crs        (time) |S1 4kB b'' b'' b'' b'' b'' b'' ... b'' b'' b'' b'' b''
    ssm        (time, lat, lon) float32 398GB dask.array<chunksize=(1, 1382, 2278), meta=np.ndarray>
    ssm_noise  (time, lat, lon) float32 398GB dask.array<chunksize=(1, 1382, 2278), meta=np.ndarray>
Attributes: (12/26)
    Conventions:               CF-1.6
    archive_facility:          VITO
    copyright:                 Copernicus Service information 2019
    geospatial_lat_max:        72.0
    geospatial_lat_min:        35.0
    geospatial_lon_max:        50.0
    ...                        ...
    region_name:               CEURO
    sensor:                    CSAR
    source:                    Derived from EO radar observations
    time_coverage_end:         2016-01-01T23:59:59Z
    time_coverage_start:       2016-01-01T00:00:00Z
    title:                     Daily Surface Soil Moisture 1km: CEURO 2016-01...

In [ ]:
# Missing dates from source - '2020-07-18', '2021-05-31' when running this for 2 years from 2020-2021

In [15]:
clms_ssm_only = clms_data.drop_vars("ssm_noise") 
clms_ssm_only

<xarray.Dataset> Size: 398GB
Dimensions:  (time: 3516, lat: 4144, lon: 6832)
Coordinates:
  * lat      (lat) float64 33kB 72.0 71.99 71.98 71.97 ... 35.02 35.01 35.0
  * lon      (lon) float64 55kB -11.0 -10.99 -10.98 -10.97 ... 49.98 49.99 50.0
  * time     (time) datetime64[ns] 28kB 2016-01-01 2016-01-02 ... 2025-09-30
Data variables:
    crs      (time) |S1 4kB b'' b'' b'' b'' b'' b'' ... b'' b'' b'' b'' b'' b''
    ssm      (time, lat, lon) float32 398GB dask.array<chunksize=(1, 1382, 2278), meta=np.ndarray>
Attributes: (12/26)
    Conventions:               CF-1.6
    archive_facility:          VITO
    copyright:                 Copernicus Service information 2019
    geospatial_lat_max:        72.0
    geospatial_lat_min:        35.0
    geospatial_lon_max:        50.0
    ...                        ...
    region_name:               CEURO
    sensor:                    CSAR
    source:                    Derived from EO radar observations
    time_coverage_end:         2016-01-01T23:59:59Z
    time_coverage_start:       2016-01-01T00:00:00Z
    title:                     Daily Surface Soil Moisture 1km: CEURO 2016-01...

In [17]:
dask.config.set(scheduler="threads", num_workers=1)

In [19]:
from xcube.core.chunk import chunk_dataset

In [22]:
clms_ssm_only_chunked = chunk_dataset(clms_ssm_only, {"time": 2, "lat": 4144, "lon": 6832})
clms_ssm_only_chunked

<xarray.Dataset> Size: 398GB
Dimensions:  (time: 3516, lat: 4144, lon: 6832)
Coordinates:
  * lat      (lat) float64 33kB 72.0 71.99 71.98 71.97 ... 35.02 35.01 35.0
  * lon      (lon) float64 55kB -11.0 -10.99 -10.98 -10.97 ... 49.98 49.99 50.0
  * time     (time) datetime64[ns] 28kB 2016-01-01 2016-01-02 ... 2025-09-30
Data variables:
    crs      (time) |S1 4kB dask.array<chunksize=(2,), meta=np.ndarray>
    ssm      (time, lat, lon) float32 398GB dask.array<chunksize=(2, 4144, 6832), meta=np.ndarray>
Attributes: (12/26)
    Conventions:               CF-1.6
    archive_facility:          VITO
    copyright:                 Copernicus Service information 2019
    geospatial_lat_max:        72.0
    geospatial_lat_min:        35.0
    geospatial_lon_max:        50.0
    ...                        ...
    region_name:               CEURO
    sensor:                    CSAR
    source:                    Derived from EO radar observations
    time_coverage_end:         2016-01-01T23:59:59Z
    time_coverage_start:       2016-01-01T00:00:00Z
    title:                     Daily Surface Soil Moisture 1km: CEURO 2016-01...

In [ ]:
%%time
irr_store.write_data(clms_ssm_only_chunked, f"clms.zarr")

In [18]:
irr_store.list_data_ids()

['clms.zarr', 'era5.zarr', 'landcover2020global.zarr']

In [19]:
irr_store.open_data("clms.zarr")

<xarray.Dataset> Size: 7GB
Dimensions:  (time: 31, lat: 4144, lon: 6832)
Coordinates:
  * lat      (lat) float64 33kB 72.0 71.99 71.98 71.97 ... 35.02 35.01 35.0
  * lon      (lon) float64 55kB -11.0 -10.99 -10.98 -10.97 ... 49.98 49.99 50.0
  * time     (time) datetime64[ns] 248B 2020-01-01 2020-01-02 ... 2020-01-31
Data variables:
    crs      (time) |S1 31B dask.array<chunksize=(31,), meta=np.ndarray>
    ssm      (time, lat, lon) float64 7GB dask.array<chunksize=(1, 1382, 2278), meta=np.ndarray>
Attributes: (12/26)
    Conventions:               CF-1.6
    archive_facility:          VITO
    copyright:                 Copernicus Service information 2020
    geospatial_lat_max:        72.0
    geospatial_lat_min:        35.0
    geospatial_lon_max:        50.0
    ...                        ...
    region_name:               CEURO
    sensor:                    CSAR
    source:                    Derived from EO radar observations
    time_coverage_end:         2020-01-01T23:59:59Z
    time_coverage_start:       2020-01-01T00:00:00Z
    title:                     Daily Surface Soil Moisture 1km: CEURO 2020-01...